In [2]:
import h5py

path = "../data/raw/reassemble/2025-01-09-13-57-17.h5"

with h5py.File(path, "r") as f:
    print(list(f.keys()))
    print(list(f["segments_info"].keys()))
    print(list(f["timestamps"].keys()))

['capture_node-camera-image', 'events', 'hama1', 'hama1_audio', 'hama2', 'hama2_audio', 'hand', 'hand_audio', 'robot_state', 'segments_info', 'timestamps']
['0', '1', '2', '3', '4']
['capture_node-camera-image', 'compensated_base_force', 'compensated_base_torque', 'events', 'gripper_efforts', 'gripper_positions', 'gripper_velocities', 'hama1', 'hama2', 'hand', 'joint_efforts', 'joint_positions', 'joint_velocities', 'measured_force', 'measured_torque', 'pose', 'velocity']


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.dataset import extract_video
video_path = extract_video(
    "../data/raw/reassemble/2025-01-09-13-57-17.h5",
    "../data/videos/",
    camera="hama1"
)

print(video_path)

../data/videos/2025-01-09-13-57-17_hama1.mp4


In [3]:
import h5py

h5_path = "../data/raw/reassemble/2025-01-09-13-57-17.h5"

with h5py.File(h5_path, "r") as f:

    segments = f["segments_info"]

    for segment_id in segments:

        segment = segments[segment_id]

        print("Segment:", segment_id)
        print("Start:", segment["start"][()])
        print("End:", segment["end"][()])
        print("Action:", segment["text"][()])
        print("Success:", segment["success"][()])
        print("-" * 40)

Segment: 0
Start: 1736427437.7950916
End: 1736427449.4950447
Action: b'No action.'
Success: True
----------------------------------------
Segment: 1
Start: 1736427449.511793
End: 1736427461.8951142
Action: b'Pick Ethernet.'
Success: True
----------------------------------------
Segment: 2
Start: 1736427461.9118068
End: 1736427485.967381
Action: b'Insert Ethernet.'
Success: False
----------------------------------------
Segment: 3
Start: 1736427485.978471
End: 1736427504.2783637
Action: b'Insert Ethernet.'
Success: False
----------------------------------------
Segment: 4
Start: 1736427504.297776
End: 1736427539.8617945
Action: b'No action.'
Success: True
----------------------------------------


In [4]:
import h5py
import numpy as np
import json
import cv2
from collections import Counter

h5_path = "../data/raw/reassemble/2025-01-09-13-57-17.h5"
video_path = "../data/videos/2025-01-09-13-57-17_hama1.mp4"

# 1. Recupera timestamp e annotazioni
annotations = []

with h5py.File(h5_path, "r") as f:

    timestamps = np.asarray(
        f["timestamps/hama1"][()]
    ).reshape(-1)

    for segment in f["segments_info"].values():

        text = segment["text"][()].decode().strip(".")

        if text == "No action":
            action, obj = "IDLE", None
        else:
            action, _, obj = text.partition(" ")

        annotations.append({
            "start": float(segment["start"][()]),
            "end": float(segment["end"][()]),
            "action": action.upper(),
            "object": obj,
            "success": bool(segment["success"][()])
        })

print("Primo timestamp:", timestamps[0])
print("Annotazioni:", annotations)


Primo timestamp: 1736427437.8118465
Annotazioni: [{'start': 1736427437.7950916, 'end': 1736427449.4950447, 'action': 'IDLE', 'object': None, 'success': True}, {'start': 1736427449.511793, 'end': 1736427461.8951142, 'action': 'PICK', 'object': 'Ethernet', 'success': True}, {'start': 1736427461.9118068, 'end': 1736427485.967381, 'action': 'INSERT', 'object': 'Ethernet', 'success': False}, {'start': 1736427485.978471, 'end': 1736427504.2783637, 'action': 'INSERT', 'object': 'Ethernet', 'success': False}, {'start': 1736427504.297776, 'end': 1736427539.8617945, 'action': 'IDLE', 'object': None, 'success': True}]


In [5]:
# 2. Carica le clip già elaborate
with open("../outputs/clips.json") as f:
    clips = json.load(f)

# 3. Verifica corrispondenza frame-timestamp
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

assert fps > 0
assert n_frames == len(timestamps)
assert np.all(np.diff(timestamps) > 0)

# 4. Assegna le ground truth
for clip in clips:

    start_idx = round(clip["start_time"] * fps)
    end_idx = round(clip["end_time"] * fps)

    start = timestamps[start_idx]

    end = (
        timestamps[end_idx]
        if end_idx < len(timestamps)
        else timestamps[-1] + np.median(np.diff(timestamps))
    )

    clip["ground_truth"] = None

    for ann in annotations:

        # Clip interamente contenuta nell'azione
        if start >= ann["start"] and end <= ann["end"]:

            clip["ground_truth"] = ann["action"]
            clip["object"] = ann["object"]
            clip["success"] = ann["success"]
            break

# 5. Salva risultati
with open("../outputs/labeled_clips.json", "w") as f:
    json.dump(clips, f, indent=2)

print(Counter(c["ground_truth"] for c in clips))

Counter({'IDLE': 28, 'INSERT': 24, None: 8, 'PICK': 7})
